# AdaFuse — Ablation Studies & Analysis

This notebook runs the key analytical experiments for the AdaFuse BTP:

| Section | Research Question | What it answers |
|---------|------------------|------------------|
| 2. Ablation Study | **RQ3** | Which components actually contribute? |
| 3. Attention Weight Analysis | **RQ2** | Does fusion weight correlate with user activity? |
| 4. Sparsity-Stratified Eval | **RQ2 + RQ4** | When does each signal dominate? |
| 5. Data Sparsity Robustness | **RQ4** | How does AdaFuse handle less training data? |
| 6. Layer Depth Analysis | — | Optimal GNN depth; does fusion reduce oversmoothing? |
| 7. Embedding Visualization | — | Qualitative structure of learned representations |
| 8. Hyperparameter Sensitivity | — | Stability across hyperparameter choices |

**Prerequisites:** Run notebooks 01 (data preprocessing), 02 (baseline training), and 03 (AdaFuse training) first.

---
## 1. Setup & Load

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import json
import pickle
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.manifold import TSNE
from collections import defaultdict

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
    'figure.facecolor': '#0e0e18',
    'axes.facecolor': '#151525',
    'savefig.facecolor': '#0e0e18',
    'savefig.bbox': 'tight',
})

COLORS = {
    'AdaFuse': '#10b981',
    'SASGCN':  '#f59e0b',
    'SASRec':  '#a78bfa',
    'LightGCN':'#00d4ff',
    'MF-BPR':  '#64748b',
    'Popularity':'#475569',
    'BERT4Rec': '#ec4899',
}

SEEDS = [42, 123, 456, 789, 1024]
METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']
FIG_DIR = '../results/figures'
os.makedirs(FIG_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
from data.dataset import load_dataset, build_graph, build_sequences
from models.lightgcn import LightGCN
from models.sasrec import SASRec
from models.sasgcn import SASGCN
from models.adafuse import AdaFuse
from evaluation.metrics import compute_metrics, per_user_metrics

In [ ]:
# ── Load preprocessed data ──
DATASET = 'amazon-beauty'  # change to 'amazon-book' for secondary dataset
DATA_DIR = f'../roadmap/data/{DATASET}'

with open(f'{DATA_DIR}/processed/train.pkl', 'rb') as f:
    train_data = pickle.load(f)
with open(f'{DATA_DIR}/processed/val.pkl', 'rb') as f:
    val_data = pickle.load(f)
with open(f'{DATA_DIR}/processed/test.pkl', 'rb') as f:
    test_data = pickle.load(f)
with open(f'{DATA_DIR}/processed/stats.json', 'r') as f:
    data_stats = json.load(f)

num_users = data_stats['num_users']
num_items = data_stats['num_items']

adj_matrix = build_graph(train_data, num_users, num_items)
user_seqs = build_sequences(train_data, max_len=50)

# Per-user interaction counts (for stratified analysis)
user_train_counts = defaultdict(int)
for u, i in train_data:
    user_train_counts[u] += 1

print(f'Users: {num_users}, Items: {num_items}')
print(f'Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}')

In [ ]:
# ── Helper: train a model from scratch and return per-user metrics ──

def train_and_evaluate(model_cls, model_kwargs, seeds=SEEDS, epochs=200, patience=50, lr=1e-3, wd=1e-4):
    """Train model across seeds, return dict of {metric: [values_per_seed]} and per-user scores."""
    all_results = {m: [] for m in METRICS}
    all_per_user = []

    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)

        model = model_cls(**model_kwargs).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

        best_val_ndcg = 0
        patience_counter = 0
        best_state = None

        for epoch in range(epochs):
            model.train()
            train_loss = model.train_step(train_data, adj_matrix, user_seqs, optimizer, device)

            if (epoch + 1) % 5 == 0:
                model.eval()
                with torch.no_grad():
                    val_metrics = compute_metrics(model, val_data, train_data,
                                                  adj_matrix, user_seqs, num_items, device)
                if val_metrics['NDCG@10'] > best_val_ndcg:
                    best_val_ndcg = val_metrics['NDCG@10']
                    patience_counter = 0
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                else:
                    patience_counter += 5
                    if patience_counter >= patience:
                        break

        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            test_metrics, pu_scores = per_user_metrics(
                model, test_data, train_data, adj_matrix, user_seqs, num_items, device
            )
        for m in METRICS:
            all_results[m].append(test_metrics[m])
        all_per_user.append(pu_scores)

    return all_results, all_per_user


def fmt(results, metric):
    vals = results[metric]
    return f'{np.mean(vals):.4f} \u00b1 {np.std(vals):.4f}'

---
## 2. Ablation Study (RQ3)

**Question:** What is the individual contribution of the Transformer branch, the GNN branch, the attention gate, and the element-wise interaction term?

We train six variants from scratch, each for 5 seeds:

| Variant | Description |
|---------|-------------|
| AdaFuse-full | Complete model |
| w/o GNN | Force α=1 → only SASRec signal |
| w/o Transformer | Force α=0 → only LightGCN signal |
| w/o attention gate | Concatenation + MLP instead of learned gate (≈ SASGCN) |
| w/o element-wise product | Gate input is [e_seq; e_col] only (2d instead of 3d) |
| Multi-head gate | Per-dimension α ∈ R^d instead of scalar |

In [ ]:
EMBED_DIM = 64
LGCN_LAYERS = 3
SASREC_LAYERS = 2
SASREC_HEADS = 2
MAX_SEQ_LEN = 50
LR = 1e-3
WD = 1e-4

base_kwargs = dict(
    num_users=num_users,
    num_items=num_items,
    embed_dim=EMBED_DIM,
    lgcn_layers=LGCN_LAYERS,
    sasrec_layers=SASREC_LAYERS,
    sasrec_heads=SASREC_HEADS,
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.2,
)

ablation_configs = {
    'AdaFuse-full':     dict(**base_kwargs, fusion='attention', gate_input='full'),
    'w/o GNN':          dict(**base_kwargs, fusion='attention', gate_input='full', force_alpha=1.0),
    'w/o Transformer':  dict(**base_kwargs, fusion='attention', gate_input='full', force_alpha=0.0),
    'w/o attn gate':    dict(**base_kwargs, fusion='concat_mlp'),
    'w/o elem product': dict(**base_kwargs, fusion='attention', gate_input='no_product'),
    'Multi-head gate':  dict(**base_kwargs, fusion='attention_multihead', gate_input='full'),
}

print(f'Running {len(ablation_configs)} ablation variants x {len(SEEDS)} seeds each...')

In [ ]:
ablation_results = {}

for name, kwargs in ablation_configs.items():
    print(f'\n{"="*60}')
    print(f'Training: {name}')
    print(f'{"="*60}')
    results, per_user = train_and_evaluate(AdaFuse, kwargs, seeds=SEEDS, lr=LR, wd=WD)
    ablation_results[name] = {'results': results, 'per_user': per_user}
    for m in METRICS:
        print(f'  {m}: {fmt(results, m)}')

In [ ]:
# ── Ablation results table ──

rows = []
for name in ablation_configs:
    r = ablation_results[name]['results']
    rows.append({
        'Variant': name,
        **{m: fmt(r, m) for m in METRICS}
    })

df_ablation = pd.DataFrame(rows)
print(df_ablation.to_string(index=False))

In [ ]:
# ── Ablation bar chart: NDCG@20 ──

fig, ax = plt.subplots(figsize=(10, 5))

names = list(ablation_configs.keys())
means = [np.mean(ablation_results[n]['results']['NDCG@20']) for n in names]
stds  = [np.std(ablation_results[n]['results']['NDCG@20'])  for n in names]
colors_bar = ['#10b981', '#a78bfa', '#00d4ff', '#f59e0b', '#94a3b8', '#ec4899']

bars = ax.bar(range(len(names)), means, yerr=stds, capsize=4,
              color=colors_bar, edgecolor='white', linewidth=0.5, alpha=0.85)

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{mean:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('NDCG@20')
ax.set_title('Ablation Study — NDCG@20 (RQ3)', fontweight='bold')
ax.grid(axis='y', alpha=0.15)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/ablation_ndcg20.png', dpi=150)
plt.show()

**Interpretation:** Compare AdaFuse-full against each ablated variant. The component whose removal causes the largest drop is the most important. If the attention gate outperforms concat+MLP, that confirms the value of adaptive per-user fusion.

---
## 3. Attention Weight Analysis (RQ2)

**Question:** How does the learned fusion weight α(u) correlate with user interaction density? Do sparse users rely more on collaborative (GNN) signal?

In [ ]:
# ── Extract alpha values from trained AdaFuse ──

# Load best AdaFuse checkpoint (seed with best val NDCG@10)
best_seed_idx = np.argmax(ablation_results['AdaFuse-full']['results']['NDCG@10'])
best_seed = SEEDS[best_seed_idx]
print(f'Using best seed: {best_seed}')

# Re-instantiate and load
model_best = AdaFuse(**ablation_configs['AdaFuse-full']).to(device)
checkpoint_path = f'../results/logs/adafuse_seed{best_seed}.pt'
model_best.load_state_dict(torch.load(checkpoint_path, map_location=device))
model_best.eval()

# Extract alpha for each test user
test_users = sorted(set(u for u, _ in test_data))
alphas = {}

with torch.no_grad():
    for u in test_users:
        alpha_u = model_best.get_fusion_alpha(u, adj_matrix, user_seqs, device)
        alphas[u] = alpha_u.item()

alpha_df = pd.DataFrame([
    {'user': u, 'alpha': alphas[u], 'num_interactions': user_train_counts[u]}
    for u in test_users
])

print(f'Alpha stats: mean={alpha_df.alpha.mean():.3f}, '
      f'std={alpha_df.alpha.std():.3f}, '
      f'min={alpha_df.alpha.min():.3f}, '
      f'max={alpha_df.alpha.max():.3f}')

In [ ]:
# ── Scatter plot: alpha vs log(interaction count) ──

fig, ax = plt.subplots(figsize=(10, 6))

log_counts = np.log1p(alpha_df['num_interactions'].values)
alpha_vals = alpha_df['alpha'].values

ax.scatter(log_counts, alpha_vals, s=8, alpha=0.3, c='#00d4ff', edgecolors='none')

# Trend line
slope, intercept, r_value, p_value, std_err = stats.linregress(log_counts, alpha_vals)
x_line = np.linspace(log_counts.min(), log_counts.max(), 100)
ax.plot(x_line, slope * x_line + intercept, color='#10b981', linewidth=2,
        label=f'Trend: r={r_value:.3f}, p={p_value:.2e}')

ax.set_xlabel('log(1 + num_interactions)')
ax.set_ylabel('Fusion weight \u03b1(u)')
ax.set_title('Attention Gate Weight vs User Activity (RQ2)', fontweight='bold')
ax.legend()
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.15)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/alpha_vs_activity_scatter.png', dpi=150)
plt.show()

# Spearman rank correlation (non-parametric, robust to outliers)
rho, p_spearman = stats.spearmanr(alpha_df['num_interactions'], alpha_df['alpha'])
print(f'Pearson  r = {r_value:.4f}, p = {p_value:.2e}')
print(f'Spearman \u03c1 = {rho:.4f}, p = {p_spearman:.2e}')

In [ ]:
# ── Box plot: alpha by user activity group ──

bins = [0, 5, 10, 20, 50, float('inf')]
labels = ['1-5', '6-10', '11-20', '21-50', '50+']
alpha_df['group'] = pd.cut(alpha_df['num_interactions'], bins=bins, labels=labels, right=True)

fig, ax = plt.subplots(figsize=(9, 5))

group_data = [alpha_df[alpha_df['group'] == g]['alpha'].values for g in labels]
bp = ax.boxplot(group_data, labels=labels, patch_artist=True, widths=0.5,
                medianprops=dict(color='white', linewidth=2),
                flierprops=dict(marker='.', markersize=3, alpha=0.4))

box_colors = ['#00d4ff', '#22d3ee', '#a78bfa', '#c084fc', '#7c3aed']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

# Overlay means
group_means = [np.mean(d) for d in group_data]
ax.scatter(range(1, len(labels) + 1), group_means, color='#10b981', s=50, zorder=5,
           marker='D', label='Mean \u03b1')

ax.set_xlabel('User Group (by training interactions)')
ax.set_ylabel('Fusion weight \u03b1(u)')
ax.set_title('\u03b1 Distribution by User Activity Level', fontweight='bold')
ax.legend()
ax.set_ylim(-0.05, 1.05)
ax.grid(axis='y', alpha=0.15)

# Annotate group sizes
for i, g in enumerate(labels):
    n = len(group_data[i])
    ax.text(i + 1, -0.08, f'n={n}', ha='center', fontsize=7, color='#64748b')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/alpha_boxplot_by_group.png', dpi=150)
plt.show()

In [ ]:
# ── Histogram of all alpha values ──

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(alpha_df['alpha'], bins=50, color='#00d4ff', alpha=0.7, edgecolor='white', linewidth=0.3)
ax.axvline(alpha_df['alpha'].mean(), color='#10b981', linewidth=2, linestyle='--',
           label=f'Mean = {alpha_df["alpha"].mean():.3f}')
ax.axvline(0.5, color='#f59e0b', linewidth=1, linestyle=':', alpha=0.6, label='\u03b1 = 0.5 (balanced)')

ax.set_xlabel('Fusion weight \u03b1(u)')
ax.set_ylabel('Number of users')
ax.set_title('Distribution of Learned \u03b1 Values', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.15)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/alpha_histogram.png', dpi=150)
plt.show()

### Interpretation (RQ2)

**Expected finding:** α should correlate positively with user interaction count:

- **Sparse users (1–5 items):** α ≈ 0.3–0.4 → rely more on GNN collaborative signal (borrowed information from similar users compensates for a short personal sequence).
- **Medium users (10–30 items):** α ≈ 0.5 → balanced fusion.
- **Active users (30+ items):** α ≈ 0.6–0.7 → rely more on Transformer sequential signal (long personal history is rich enough to model intent directly).

If the correlation is statistically significant (p < 0.05), this validates our core hypothesis: the attention gate learns a meaningful, interpretable fusion strategy that adapts to each user's data availability.

---
## 4. Sparsity-Stratified Evaluation (RQ2 + RQ4)

**Question:** How does each model perform across different user activity levels? Does AdaFuse's advantage concentrate in any particular group?

In [ ]:
# ── Group test users by training interaction count ──

GROUP_BINS = [0, 5, 10, 20, 50, float('inf')]
GROUP_LABELS = ['1-5', '6-10', '11-20', '21-50', '50+']

test_users_all = sorted(set(u for u, _ in test_data))
user_groups = {}
for u in test_users_all:
    cnt = user_train_counts[u]
    for j in range(len(GROUP_BINS) - 1):
        if GROUP_BINS[j] < cnt <= GROUP_BINS[j + 1]:
            user_groups[u] = GROUP_LABELS[j]
            break

group_sizes = {g: sum(1 for u in user_groups.values() if u == g) for g in GROUP_LABELS}
print('User group sizes:')
for g, n in group_sizes.items():
    print(f'  {g:>5s}: {n:,} users')

In [ ]:
# ── Compute per-group metrics for each model ──
# We use per-user scores from the best seed of each model.

MODELS_TO_COMPARE = ['LightGCN', 'SASRec', 'SASGCN', 'AdaFuse']

# Load per-user scores — these should have been saved during notebook 02/03 training
# Format: dict mapping user_id -> dict of {metric: value}
model_per_user = {}
for model_name in MODELS_TO_COMPARE:
    path = f'../results/logs/{model_name.lower()}_per_user_scores.pkl'
    with open(path, 'rb') as f:
        model_per_user[model_name] = pickle.load(f)

# Compute group-level metrics
stratified = {model: {g: {m: [] for m in METRICS} for g in GROUP_LABELS}
              for model in MODELS_TO_COMPARE}

for model_name in MODELS_TO_COMPARE:
    pu = model_per_user[model_name]
    for u in test_users_all:
        g = user_groups.get(u)
        if g is None or u not in pu:
            continue
        for m in METRICS:
            stratified[model_name][g][m].append(pu[u][m])

# Average per group
strat_means = {model: {g: {m: np.mean(stratified[model][g][m]) if stratified[model][g][m] else 0
                            for m in METRICS}
                        for g in GROUP_LABELS}
               for model in MODELS_TO_COMPARE}

In [ ]:
# ── Table: models x groups x NDCG@20 ──

rows = []
for model in MODELS_TO_COMPARE:
    row = {'Model': model}
    for g in GROUP_LABELS:
        row[g] = f"{strat_means[model][g]['NDCG@20']:.4f}"
    rows.append(row)

df_strat = pd.DataFrame(rows)
print('NDCG@20 by User Activity Group')
print(df_strat.to_string(index=False))

In [ ]:
# ── Line chart: NDCG@20 vs user group, one line per model ──

fig, ax = plt.subplots(figsize=(10, 6))

x = range(len(GROUP_LABELS))

for model in MODELS_TO_COMPARE:
    vals = [strat_means[model][g]['NDCG@20'] for g in GROUP_LABELS]
    ax.plot(x, vals, marker='o', linewidth=2.5, markersize=8,
            label=model, color=COLORS.get(model, '#888'))

ax.set_xticks(x)
ax.set_xticklabels(GROUP_LABELS)
ax.set_xlabel('User Group (by # training interactions)')
ax.set_ylabel('NDCG@20')
ax.set_title('Performance by User Activity Level (RQ2 + RQ4)', fontweight='bold')
ax.legend()
ax.grid(alpha=0.15)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/stratified_ndcg20_by_group.png', dpi=150)
plt.show()

In [ ]:
# ── Heatmap: AdaFuse improvement over best single-signal baseline ──

improvement = np.zeros((len(METRICS), len(GROUP_LABELS)))

for mi, m in enumerate(METRICS):
    for gi, g in enumerate(GROUP_LABELS):
        adafuse_val = strat_means['AdaFuse'][g][m]
        best_single = max(strat_means['LightGCN'][g][m], strat_means['SASRec'][g][m])
        if best_single > 0:
            improvement[mi, gi] = ((adafuse_val - best_single) / best_single) * 100
        else:
            improvement[mi, gi] = 0

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(improvement, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            xticklabels=GROUP_LABELS, yticklabels=METRICS, ax=ax,
            linewidths=0.5, linecolor='#1a1a2e',
            annot_kws={'fontsize': 10, 'fontweight': 'bold'})

ax.set_xlabel('User Group')
ax.set_title('AdaFuse Improvement (%) over Best Single-Signal Baseline', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/adafuse_improvement_heatmap.png', dpi=150)
plt.show()

### Interpretation

If AdaFuse's improvement is largest for medium-sparsity users (6–20 interactions) and smaller for the extremes, it suggests:

- **Very sparse users (1–5):** Both signals are weak — short sequence AND limited graph connectivity. Improvement is modest.
- **Medium users (6–20):** The sweet spot — enough data for both SASRec and LightGCN to produce complementary signals. Fusion yields the largest gain.
- **Very active users (50+):** SASRec alone is already strong with such long sequences, so the marginal value of GNN signal shrinks.

---
## 5. Data Sparsity Robustness (RQ4)

**Question:** How does each model degrade when we progressively reduce training data?

In [ ]:
# ── Subsample training data at different fractions ──

FRACTIONS = [1.0, 0.75, 0.50, 0.25]
SPARSITY_SEEDS = [42, 123, 456]  # 3 seeds for faster runtime

def subsample_train(train_data, fraction, seed):
    rng = np.random.RandomState(seed)
    n = len(train_data)
    k = int(n * fraction)
    indices = rng.choice(n, size=k, replace=False)
    return [train_data[i] for i in sorted(indices)]

sparsity_results = {model: {frac: {m: [] for m in METRICS}
                            for frac in FRACTIONS}
                    for model in MODELS_TO_COMPARE}

for frac in FRACTIONS:
    print(f'\n{"="*60}')
    print(f'Training fraction: {frac:.0%}')
    print(f'{"="*60}')

    for seed in SPARSITY_SEEDS:
        sub_train = subsample_train(train_data, frac, seed)
        sub_adj = build_graph(sub_train, num_users, num_items)
        sub_seqs = build_sequences(sub_train, max_len=MAX_SEQ_LEN)

        for model_name in MODELS_TO_COMPARE:
            if model_name == 'LightGCN':
                cls, kw = LightGCN, dict(num_users=num_users, num_items=num_items,
                                          embed_dim=EMBED_DIM, num_layers=LGCN_LAYERS)
            elif model_name == 'SASRec':
                cls, kw = SASRec, dict(num_items=num_items, embed_dim=EMBED_DIM,
                                       num_layers=SASREC_LAYERS, num_heads=SASREC_HEADS,
                                       max_seq_len=MAX_SEQ_LEN)
            elif model_name == 'SASGCN':
                cls, kw = SASGCN, dict(num_users=num_users, num_items=num_items,
                                       embed_dim=EMBED_DIM)
            else:  # AdaFuse
                cls, kw = AdaFuse, ablation_configs['AdaFuse-full']

            torch.manual_seed(seed)
            np.random.seed(seed)
            model = cls(**kw).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)

            best_val_ndcg = 0
            patience_counter = 0
            best_state = None

            for epoch in range(200):
                model.train()
                model.train_step(sub_train, sub_adj, sub_seqs, optimizer, device)
                if (epoch + 1) % 10 == 0:
                    model.eval()
                    with torch.no_grad():
                        val_m = compute_metrics(model, val_data, sub_train,
                                                sub_adj, sub_seqs, num_items, device)
                    if val_m['NDCG@10'] > best_val_ndcg:
                        best_val_ndcg = val_m['NDCG@10']
                        patience_counter = 0
                        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    else:
                        patience_counter += 10
                        if patience_counter >= 50:
                            break

            if best_state:
                model.load_state_dict(best_state)
            model.eval()
            with torch.no_grad():
                test_m = compute_metrics(model, test_data, sub_train,
                                         sub_adj, sub_seqs, num_items, device)
            for m in METRICS:
                sparsity_results[model_name][frac][m].append(test_m[m])

            print(f'  {model_name:>10s} @ {frac:.0%}: NDCG@20={test_m["NDCG@20"]:.4f}')

In [ ]:
# ── Line chart: NDCG@20 vs training fraction ──

fig, ax = plt.subplots(figsize=(9, 6))

for model in MODELS_TO_COMPARE:
    means = [np.mean(sparsity_results[model][f]['NDCG@20']) for f in FRACTIONS]
    stds  = [np.std(sparsity_results[model][f]['NDCG@20'])  for f in FRACTIONS]
    pcts  = [int(f * 100) for f in FRACTIONS]
    ax.errorbar(pcts, means, yerr=stds, marker='o', linewidth=2.5, markersize=8,
                capsize=4, label=model, color=COLORS.get(model, '#888'))

ax.set_xlabel('Training Data Fraction (%)')
ax.set_ylabel('NDCG@20')
ax.set_title('Robustness Under Data Sparsity (RQ4)', fontweight='bold')
ax.set_xticks([int(f * 100) for f in FRACTIONS])
ax.legend()
ax.grid(alpha=0.15)
ax.invert_xaxis()  # 100% on left, 25% on right

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/sparsity_robustness_ndcg20.png', dpi=150)
plt.show()

### Interpretation

A model that degrades gracefully under sparsity is practically more useful. We expect:

- **SASRec:** Drops sharply at low fractions — fewer interactions = shorter sequences = weaker signal.
- **LightGCN:** More robust (graph connectivity persists even with fewer edges per user).
- **AdaFuse:** Should be the most robust — it can shift weight toward whichever signal survives the reduction better.

---
## 6. Layer Depth Analysis

How does the number of LightGCN layers in AdaFuse affect performance? Does the attention gate help manage oversmoothing at higher depths?

In [ ]:
# ── Vary LightGCN layers: 1 through 5, SASRec fixed at 2 ──

LAYER_RANGE = [1, 2, 3, 4, 5]
LAYER_SEEDS = [42, 123, 456]

layer_results_adafuse = {L: {m: [] for m in METRICS} for L in LAYER_RANGE}
layer_results_lgcn    = {L: {m: [] for m in METRICS} for L in LAYER_RANGE}

for L in LAYER_RANGE:
    print(f'\n--- LightGCN layers = {L} ---')
    for seed in LAYER_SEEDS:
        # AdaFuse with L layers
        kw = dict(**base_kwargs, fusion='attention', gate_input='full')
        kw['lgcn_layers'] = L
        res_ada, _ = train_and_evaluate(AdaFuse, kw, seeds=[seed], lr=LR, wd=WD)
        for m in METRICS:
            layer_results_adafuse[L][m].extend(res_ada[m])

        # Standalone LightGCN with L layers (for comparison)
        kw_lgcn = dict(num_users=num_users, num_items=num_items,
                       embed_dim=EMBED_DIM, num_layers=L)
        res_lgcn, _ = train_and_evaluate(LightGCN, kw_lgcn, seeds=[seed], lr=LR, wd=WD)
        for m in METRICS:
            layer_results_lgcn[L][m].extend(res_lgcn[m])

In [ ]:
# ── Line chart: NDCG@20 vs GNN depth ──

fig, ax = plt.subplots(figsize=(8, 5))

ada_means = [np.mean(layer_results_adafuse[L]['NDCG@20']) for L in LAYER_RANGE]
ada_stds  = [np.std(layer_results_adafuse[L]['NDCG@20'])  for L in LAYER_RANGE]
lgcn_means = [np.mean(layer_results_lgcn[L]['NDCG@20']) for L in LAYER_RANGE]
lgcn_stds  = [np.std(layer_results_lgcn[L]['NDCG@20'])  for L in LAYER_RANGE]

ax.errorbar(LAYER_RANGE, ada_means, yerr=ada_stds, marker='o', linewidth=2.5,
            markersize=8, capsize=4, label='AdaFuse', color=COLORS['AdaFuse'])
ax.errorbar(LAYER_RANGE, lgcn_means, yerr=lgcn_stds, marker='s', linewidth=2.5,
            markersize=8, capsize=4, label='LightGCN', color=COLORS['LightGCN'])

ax.set_xlabel('Number of LightGCN Layers')
ax.set_ylabel('NDCG@20')
ax.set_title('Impact of GNN Depth on Performance', fontweight='bold')
ax.set_xticks(LAYER_RANGE)
ax.legend()
ax.grid(alpha=0.15)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/layer_depth_analysis.png', dpi=150)
plt.show()

### Interpretation

Classic LightGCN peaks at 3–4 layers and drops at layer 5 due to oversmoothing (embeddings converge to the same value). If AdaFuse degrades less at depth 5, the attention gate may compensate by shifting weight away from the oversmoothed GNN signal toward the Transformer.

---
## 7. Embedding Visualization

t-SNE visualization of user embeddings from three models, colored by user activity level.

In [ ]:
# ── Extract embeddings from each model ──

def extract_user_embeddings(model, users, adj_matrix, user_seqs, device):
    model.eval()
    with torch.no_grad():
        embeds = model.get_user_embeddings(users, adj_matrix, user_seqs, device)
    return embeds.cpu().numpy()

sample_users = test_users_all[:2000]  # subsample for t-SNE speed
sample_counts = np.array([user_train_counts[u] for u in sample_users])

# Load each model's best checkpoint
embed_dict = {}
for model_name in ['LightGCN', 'SASRec', 'AdaFuse']:
    path = f'../results/logs/{model_name.lower()}_best.pt'
    if model_name == 'LightGCN':
        m = LightGCN(num_users=num_users, num_items=num_items,
                     embed_dim=EMBED_DIM, num_layers=LGCN_LAYERS).to(device)
    elif model_name == 'SASRec':
        m = SASRec(num_items=num_items, embed_dim=EMBED_DIM,
                   num_layers=SASREC_LAYERS, num_heads=SASREC_HEADS,
                   max_seq_len=MAX_SEQ_LEN).to(device)
    else:
        m = AdaFuse(**ablation_configs['AdaFuse-full']).to(device)

    m.load_state_dict(torch.load(path, map_location=device))
    embed_dict[model_name] = extract_user_embeddings(m, sample_users, adj_matrix, user_seqs, device)
    print(f'{model_name}: embedding shape = {embed_dict[model_name].shape}')

In [ ]:
# ── t-SNE + 3 subplots side by side ──

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

log_counts = np.log1p(sample_counts)
vmin, vmax = log_counts.min(), log_counts.max()

for ax, (model_name, embeds) in zip(axes, embed_dict.items()):
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    proj = tsne.fit_transform(embeds)

    sc = ax.scatter(proj[:, 0], proj[:, 1], c=log_counts, cmap='coolwarm',
                    s=6, alpha=0.6, vmin=vmin, vmax=vmax, edgecolors='none')
    ax.set_title(model_name, fontweight='bold', fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])

# Shared colorbar
cbar = fig.colorbar(sc, ax=axes, orientation='vertical', fraction=0.02, pad=0.02)
cbar.set_label('log(1 + interactions)', fontsize=10)

fig.suptitle('t-SNE of User Embeddings (colored by activity)', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/embedding_tsne_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation

Look for:

- **LightGCN:** Users clustered by graph connectivity (community structure). Sparse users may be poorly placed.
- **SASRec:** Users clustered by sequential pattern similarity. Users with similar recent behavior are close.
- **AdaFuse:** Should show a more structured space — users are organized by both community AND behavioral similarity, with smoother transitions between sparse and dense users.

---
## 8. Hyperparameter Sensitivity

How sensitive is AdaFuse to key hyperparameter choices?

In [ ]:
# ── Learning rate sensitivity ──

LR_RANGE = [5e-4, 1e-3, 2e-3]
HP_SEEDS = [42, 123, 456]

lr_results = {lr: {m: [] for m in METRICS} for lr in LR_RANGE}

for lr_val in LR_RANGE:
    print(f'LR = {lr_val}')
    res, _ = train_and_evaluate(AdaFuse, ablation_configs['AdaFuse-full'],
                                 seeds=HP_SEEDS, lr=lr_val, wd=WD)
    lr_results[lr_val] = res
    print(f'  NDCG@20: {fmt(res, "NDCG@20")}')

In [ ]:
# ── GNN layer sensitivity ──

GNN_LAYERS_RANGE = [2, 3, 4]
gnn_layer_results = {L: {m: [] for m in METRICS} for L in GNN_LAYERS_RANGE}

for L in GNN_LAYERS_RANGE:
    kw = dict(**base_kwargs, fusion='attention', gate_input='full')
    kw['lgcn_layers'] = L
    res, _ = train_and_evaluate(AdaFuse, kw, seeds=HP_SEEDS, lr=LR, wd=WD)
    gnn_layer_results[L] = res
    print(f'GNN layers={L}: NDCG@20 = {fmt(res, "NDCG@20")}')

In [ ]:
# ── SASRec layer sensitivity ──

SA_LAYERS_RANGE = [1, 2]
sa_layer_results = {L: {m: [] for m in METRICS} for L in SA_LAYERS_RANGE}

for L in SA_LAYERS_RANGE:
    kw = dict(**base_kwargs, fusion='attention', gate_input='full')
    kw['sasrec_layers'] = L
    res, _ = train_and_evaluate(AdaFuse, kw, seeds=HP_SEEDS, lr=LR, wd=WD)
    sa_layer_results[L] = res
    print(f'SASRec layers={L}: NDCG@20 = {fmt(res, "NDCG@20")}')

In [ ]:
# ── Batch size sensitivity ──

BS_RANGE = [256, 512, 1024]
bs_results = {bs: {m: [] for m in METRICS} for bs in BS_RANGE}

for bs in BS_RANGE:
    kw = dict(**base_kwargs, fusion='attention', gate_input='full')
    kw['batch_size'] = bs
    res, _ = train_and_evaluate(AdaFuse, kw, seeds=HP_SEEDS, lr=LR, wd=WD)
    bs_results[bs] = res
    print(f'Batch size={bs}: NDCG@20 = {fmt(res, "NDCG@20")}')

In [ ]:
# ── Combined sensitivity plot: 2x2 subplots ──

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Learning rate
ax = axes[0, 0]
means = [np.mean(lr_results[lr]['NDCG@20']) for lr in LR_RANGE]
stds  = [np.std(lr_results[lr]['NDCG@20']) for lr in LR_RANGE]
ax.bar(range(len(LR_RANGE)), means, yerr=stds, capsize=4,
       color=['#10b981','#00d4ff','#a78bfa'], alpha=0.8, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(LR_RANGE)))
ax.set_xticklabels([f'{lr:.0e}' for lr in LR_RANGE])
ax.set_title('Learning Rate', fontweight='bold')
ax.set_ylabel('NDCG@20')
ax.grid(axis='y', alpha=0.15)

# GNN layers
ax = axes[0, 1]
means = [np.mean(gnn_layer_results[L]['NDCG@20']) for L in GNN_LAYERS_RANGE]
stds  = [np.std(gnn_layer_results[L]['NDCG@20']) for L in GNN_LAYERS_RANGE]
ax.bar(range(len(GNN_LAYERS_RANGE)), means, yerr=stds, capsize=4,
       color=['#00d4ff','#10b981','#a78bfa'], alpha=0.8, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(GNN_LAYERS_RANGE)))
ax.set_xticklabels([str(L) for L in GNN_LAYERS_RANGE])
ax.set_title('LightGCN Layers', fontweight='bold')
ax.set_ylabel('NDCG@20')
ax.grid(axis='y', alpha=0.15)

# SASRec layers
ax = axes[1, 0]
means = [np.mean(sa_layer_results[L]['NDCG@20']) for L in SA_LAYERS_RANGE]
stds  = [np.std(sa_layer_results[L]['NDCG@20']) for L in SA_LAYERS_RANGE]
ax.bar(range(len(SA_LAYERS_RANGE)), means, yerr=stds, capsize=4,
       color=['#a78bfa','#ec4899'], alpha=0.8, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(SA_LAYERS_RANGE)))
ax.set_xticklabels([str(L) for L in SA_LAYERS_RANGE])
ax.set_title('SASRec Layers', fontweight='bold')
ax.set_ylabel('NDCG@20')
ax.grid(axis='y', alpha=0.15)

# Batch size
ax = axes[1, 1]
means = [np.mean(bs_results[bs]['NDCG@20']) for bs in BS_RANGE]
stds  = [np.std(bs_results[bs]['NDCG@20']) for bs in BS_RANGE]
ax.bar(range(len(BS_RANGE)), means, yerr=stds, capsize=4,
       color=['#f59e0b','#10b981','#00d4ff'], alpha=0.8, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(BS_RANGE)))
ax.set_xticklabels([str(bs) for bs in BS_RANGE])
ax.set_title('Batch Size', fontweight='bold')
ax.set_ylabel('NDCG@20')
ax.grid(axis='y', alpha=0.15)

fig.suptitle('Hyperparameter Sensitivity (AdaFuse)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/hyperparameter_sensitivity.png', dpi=150)
plt.show()

---
## 9. Summary of Findings

### RQ1 — Does adaptive fusion beat static fusion?

*(Fill in after running experiments)*

Compare AdaFuse-full vs SASGCN (concat+MLP fusion) from the ablation table.

### RQ2 — Does the fusion weight correlate with user activity?

*(Fill in after running experiments)*

Report Pearson r and Spearman rho from Section 3. Describe the trend in the box plot.

### RQ3 — Which components contribute most?

*(Fill in after running experiments)*

Rank the ablation variants by drop from AdaFuse-full. Identify the single most critical component.

### RQ4 — How robust is AdaFuse under data sparsity?

*(Fill in after running experiments)*

Describe the degradation curve from Section 5. Which model is most robust at 25% data?

### Key Takeaway

*(Fill in after running experiments)*

One-sentence contribution: "AdaFuse's attention-based fusion of SASRec and LightGCN representations outperforms static fusion by X% and demonstrates that the relative importance of sequential vs. collaborative signal varies systematically with user interaction density."